# Data Drift Detection for Churn Prediction Model

## Purpose
Detect distribution shifts between training data and new incoming data to ensure model reliability.

## Why Drift Detection Matters
- **Model degradation**: When data distribution changes, model accuracy drops
- **Early warning**: Detect issues before they impact business decisions
- **Retraining triggers**: Know when to retrain the model

## Approach
1. **Numeric features**: Compare mean/std shifts, use Kolmogorov-Smirnov test
2. **Categorical features**: Compare proportion shifts, use Chi-square test
3. **Severity levels**: LOW (monitor) / MEDIUM (investigate) / HIGH (action needed)

## Test Cases
- `baseline_large.csv` (500 samples) - Should show NO/LOW drift (matches training distribution, reduced sampling noise)
- `drifted_data.csv` - Should detect SIGNIFICANT drift

In [1]:
import pandas as pd
import numpy as np
import json
from scipy import stats
from typing import Dict, List, Any, Tuple
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Load Training Reference Statistics
These were saved during model training and serve as our baseline for comparison.

In [2]:
# Load training statistics baseline
with open('../data/processed/training_reference_stats.json', 'r') as f:
    training_stats = json.load(f)

print("Training Reference Statistics Loaded")
print("=" * 50)
print(f"Training samples: {training_stats['metadata']['n_samples']}")
print(f"Numeric features: {list(training_stats['numeric_features'].keys())}")
print(f"Categorical features: {list(training_stats['categorical_features'].keys())}")
print(f"Timestamp: {training_stats['metadata']['timestamp']}")

Training Reference Statistics Loaded
Training samples: 5625
Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Timestamp: 2026-03-22T15:19:32.654260


## Drift Detection Functions

In [4]:
# Load baseline test data (500 samples - larger for reliable testing)
baseline_df = pd.read_csv('../data/user_testing/baseline_large.csv')
print(f"Loaded {len(baseline_df)} baseline test samples")
print(f"Expected: Minimal drift (data matches training distribution)")


Loaded 500 baseline test samples
Expected: Minimal drift (data matches training distribution)


## Load Test Data (Baseline & Drifted)
Use larger baseline dataset (500 samples) for more reliable drift detection with reduced sampling noise.

In [3]:
# Import drift detector from production module
import sys
sys.path.insert(0, '../src')

from drift_detector import DriftDetector

# Initialize detector with training statistics
detector = DriftDetector(training_stats)
print("DriftDetector imported from production module!")
print(f"  Monitoring {len(detector.numeric_features)} numeric features")
print(f"  Monitoring {len(detector.categorical_features)} categorical features")


DriftDetector imported from production module!
  Monitoring 4 numeric features
  Monitoring 15 categorical features


In [5]:
# Run drift detection on baseline data
baseline_report = detector.detect_drift(baseline_df)
detector.print_report(baseline_report)


DRIFT DETECTION REPORT - ✓ OK
Timestamp: 2026-03-30T14:36:39.985038

Summary:
  Samples analyzed: 500
  Total features: 19
  Features drifted: 0
    - HIGH severity: 0
    - MEDIUM severity: 0

----------------------------------------------------------------------------------------------------
NUMERIC FEATURES (Z-Score + KS Test + PSI)
----------------------------------------------------------------------------------------------------
  Feature          Z-Score  KS Stat   KS p-val      PSI     Severity
----------------------------------------------------------------------------------------------------
  SeniorCitizen      0.027    0.518     0.0000   0.0027          LOW
  tenure             0.025    0.137     0.0000   0.0025          LOW
  MonthlyCharges     0.013    0.128     0.0000   0.0013          LOW
  TotalCharges       0.022    0.184     0.0000   0.0022          LOW

----------------------------------------------------------------------------------------------------
CATEGORICAL 

In [6]:
# Compact baseline summary (large baseline)
print("\n" + "=" * 70)
print("BASELINE LARGE SUMMARY")
print("=" * 70)
print(f"Overall Status: {baseline_report['overall_status']}")
print(f"High Severity: {baseline_report['summary']['high_severity']}")
print(f"Medium Severity: {baseline_report['summary']['medium_severity']}")
print(f"Numeric Drifted: {baseline_report['summary']['numeric_drifted']}")
print(f"Categorical Drifted: {baseline_report['summary']['categorical_drifted']}")

high_or_med = []
for feature, info in baseline_report['numeric_features'].items():
    if info.get('severity') in ('MEDIUM', 'HIGH'):
        high_or_med.append((feature, info.get('severity')))
for feature, info in baseline_report['categorical_features'].items():
    if info.get('severity') in ('MEDIUM', 'HIGH'):
        high_or_med.append((feature, info.get('severity')))

if high_or_med:
    print("\nFeatures with MEDIUM/HIGH severity:")
    for feat, sev in high_or_med:
        print(f"  - {feat}: {sev}")
else:
    print("\nNo MEDIUM/HIGH severity features. ✅")
print("=" * 70)


BASELINE LARGE SUMMARY
Overall Status: OK
High Severity: 0
Medium Severity: 0
Numeric Drifted: 0
Categorical Drifted: 0

No MEDIUM/HIGH severity features. ✅


## Test 2: Drifted Data (Should Detect SIGNIFICANT Drift)
The drifted data was specifically created with distribution shifts:

In [11]:
# Load drifted test data
drifted_df = pd.read_csv('../data/user_testing/drifted_data.csv')
print(f"Loaded {len(drifted_df)} drifted test samples")
print(f"\nKey drift indicators in this data:")
print(f"  - SeniorCitizen rate: {drifted_df['SeniorCitizen'].mean()*100:.1f}%")
print(f"  - Average tenure: {drifted_df['tenure'].mean():.1f} months")
print(f"  - Fiber optic rate: {(drifted_df['InternetService']=='Fiber optic').mean()*100:.1f}%")
drifted_df.head()

Loaded 10 drifted test samples

Key drift indicators in this data:
  - SeniorCitizen rate: 80.0%
  - Average tenure: 1.5 months
  - Fiber optic rate: 90.0%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,DRIFT-001,Female,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.50,95.50,Yes
1,DRIFT-002,Female,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,105.00,105.00,Yes
2,DRIFT-003,Female,1,No,No,2,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.80,199.60,Yes
3,DRIFT-004,Female,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,110.25,110.25,Yes
4,DRIFT-005,Female,1,No,No,3,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,103.50,310.50,Yes


In [12]:
# Run drift detection on drifted data
drifted_report = detector.detect_drift(drifted_df)
detector.print_report(drifted_report)


DRIFT DETECTION REPORT - ✗ CRITICAL
Timestamp: 2026-03-30T14:31:01.301749

Summary:
  Samples analyzed: 10
  Total features: 19
  Features drifted: 19
    - HIGH severity: 14
    - MEDIUM severity: 5

----------------------------------------------------------------------------------------------------
NUMERIC FEATURES (Z-Score + KS Test + PSI)
----------------------------------------------------------------------------------------------------
  Feature          Z-Score  KS Stat   KS p-val      PSI     Severity
----------------------------------------------------------------------------------------------------
! SeniorCitizen      1.733    0.789     0.0000   0.1733         HIGH
! tenure             1.266    0.886     0.0000   0.1266         HIGH
! MonthlyCharges     1.186    0.796     0.0000   0.1186         HIGH
! TotalCharges       0.945    0.809     0.0000   0.0945         HIGH

----------------------------------------------------------------------------------------------------
CATEG

---
## ✅ Drift Detection Complete

For comprehensive visualizations and analysis, see: **04_drift_visualization.ipynb**

The visualization notebook includes:
- KDE distribution plots and violin plots
- Drift metrics heatmaps
- Numeric and categorical feature comparison
- Performance benchmarking
- Feature-by-feature summary tables